# model-clinic Training Monitor

Most model health problems originate during training. By the time you examine a finished checkpoint, you're doing post-mortem analysis. `ClinicMonitor` lets you catch problems **while they are happening**.

This notebook covers:

1. Basic `ClinicMonitor` usage in a plain PyTorch training loop
2. Detecting gradient explosion mid-training
3. Using `ClinicTrainerCallback` with HuggingFace `Trainer`
4. Interpreting monitor alerts
5. Using the monitor summary after training

**Requirements**: `pip install model-clinic`  
For HuggingFace callback: `pip install model-clinic[hf]`

All examples use tiny toy models — no GPU or real data required.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model_clinic import ClinicMonitor, MonitorAlert, MonitorSummary

## 1. Build a Toy Model

We need a real `nn.Module` (not just a state dict) because `ClinicMonitor` attaches forward hooks to detect layer collapse and reads `.grad` tensors during training.

In [ ]:
class TinyMLP(nn.Module):
    """Small MLP for demonstration. Mimics a 3-layer transformer MLP block."""

    def __init__(self, dim=64, hidden=256):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x):
        return self.layers(x)


model = TinyMLP(dim=64, hidden=256)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## 2. Attach ClinicMonitor

`ClinicMonitor` takes the model at construction time. It:
- Snapshots initial weight norms for divergence detection
- Registers forward hooks for layer-collapse detection
- Tracks gradient norms and loss at each `step()` call

Key parameters:
- `log_every` — how many steps between health checks (default: 100)
- `alert_on` — filter to specific conditions (default: all)
- `log_fn` — optional callback `(step, alerts)` for custom handling

In [ ]:
# Custom alert handler — print alerts as they fire
def on_alert(step, alerts):
    for alert in alerts:
        print(f"  ALERT step {step:4d}: [{alert.severity}] {alert.condition} — {alert.message}")


monitor = ClinicMonitor(
    model,
    log_every=50,          # Check every 50 steps
    alert_on=None,         # All conditions
    log_fn=on_alert,       # Print alerts live
)

print("Monitor attached. Forward hooks registered.")
print(f"Watching {len(monitor._initial_norms)} parameters for weight divergence.")

## 3. Normal Training Loop (No Problems)

First, a clean training loop to establish a baseline. Call `monitor.step(step, loss=loss)` once per training step.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print("Training for 200 steps (normal, no alerts expected)...")
print()

all_alerts = []
for step in range(200):
    # Fake batch: random input, random target
    x = torch.randn(8, 64)          # batch_size=8, input_dim=64
    target = torch.randn(8, 64)

    optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    # Tell the monitor about this step
    alerts = monitor.step(step, loss=loss.item())
    all_alerts.extend(alerts)

print(f"Training complete. Total alerts: {len(all_alerts)}")

## 4. Simulate Gradient Explosion

Now we'll deliberately trigger gradient explosion by using a huge learning rate. The monitor should fire `gradient_explosion` alerts.

In [ ]:
import copy

# Fresh model and monitor for the explosion demo
exploding_model = TinyMLP(dim=64, hidden=256)

explosion_alerts = []
def capture_alerts(step, alerts):
    explosion_alerts.extend(alerts)
    for a in alerts:
        print(f"  ALERT step {step:4d}: [{a.severity:5s}] {a.condition} — {a.message}")

explode_monitor = ClinicMonitor(
    exploding_model,
    log_every=10,
    log_fn=capture_alerts,
)

# Huge LR to force gradient explosion
bad_optimizer = optim.SGD(exploding_model.parameters(), lr=100.0)
criterion = nn.MSELoss()

print("Training with LR=100 to trigger gradient explosion...")
print()

for step in range(100):
    x = torch.randn(8, 64)
    target = torch.randn(8, 64)

    bad_optimizer.zero_grad()
    try:
        output = exploding_model(x)
        loss = criterion(output, target)
        if torch.isfinite(loss):
            loss.backward()
            bad_optimizer.step()
            explode_monitor.step(step, loss=loss.item())
        else:
            print(f"  Step {step}: loss is NaN/Inf, stopping")
            break
    except Exception as e:
        print(f"  Step {step}: exception {e}, stopping")
        break

print()
print(f"Total explosion alerts: {len(explosion_alerts)}")

## 5. Monitor Summary

After training (or at any point), call `monitor.summary()` to get full statistics. The summary includes the gradient norm history, loss history, and dead neuron tracking over the entire run.

In [ ]:
summary = monitor.summary()

print("=== Monitor Summary (clean training run) ===")
print(f"Total steps       : {summary.total_steps}")
print(f"Total alerts      : {summary.total_alerts}")
print()

if summary.alerts_by_condition:
    print("Alerts by condition:")
    for cond, count in sorted(summary.alerts_by_condition.items()):
        print(f"  {cond}: {count}")
else:
    print("No alerts fired during clean training run.")

if summary.gradient_norm_history:
    norms = [v for _, v in summary.gradient_norm_history]
    print(f"\nGradient norm stats:")
    print(f"  Min : {min(norms):.4f}")
    print(f"  Max : {max(norms):.4f}")
    print(f"  Mean: {sum(norms)/len(norms):.4f}")

In [ ]:
# Optional: plot gradient norm history if matplotlib is available
try:
    import matplotlib.pyplot as plt

    steps = [s for s, _ in summary.gradient_norm_history]
    norms = [v for _, v in summary.gradient_norm_history]

    plt.figure(figsize=(10, 4))
    plt.plot(steps, norms, linewidth=1.5)
    plt.xlabel("Step")
    plt.ylabel("Max Gradient Norm")
    plt.title("Gradient Norm History (clean run)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("(Gradient norm plot shown above)")
except ImportError:
    print("matplotlib not available — skipping plot.")
    # Print a text sparkline instead
    if summary.gradient_norm_history:
        norms = [v for _, v in summary.gradient_norm_history]
        mn, mx = min(norms), max(norms)
        if mx > mn:
            bars = ['▁▂▃▄▅▆▇█'[int(7*(v-mn)/(mx-mn))] for v in norms]
            print("Gradient norm history: " + ''.join(bars))

## 6. Alert Conditions Reference

| Condition | Severity | Trigger |
|-----------|----------|--------|
| `gradient_explosion` | ERROR | max grad norm > 100 **or** > 10x rolling average |
| `gradient_vanishing` | WARN  | max grad norm < 1e-7 |
| `neuron_death`       | ERROR | >5% of neurons have near-zero weight AND near-zero grad |
| `loss_spike`         | WARN  | loss > 2x rolling average |
| `weight_divergence`  | WARN  | weight norm has moved >10x from initial value |
| `layer_collapse`     | ERROR | layer output std < 1e-6 (all outputs the same) |

### Recommended responses

| Alert | Action |
|-------|--------|
| `gradient_explosion` | Stop training, lower LR, add gradient clipping (`max_norm=1.0`) |
| `gradient_vanishing` | Check LR schedule; consider warmup or higher LR |
| `neuron_death`       | Examine dead layers post-training; apply `dead_neurons` treatment |
| `loss_spike`         | Check data pipeline for corrupted batches |
| `weight_divergence`  | Investigate specific parameter; may need weight decay |
| `layer_collapse`     | Critical — check for gradient issues or bad initialization |

## 7. HuggingFace Trainer Callback

`ClinicTrainerCallback` plugs directly into HuggingFace `Trainer`. No changes to your training loop needed.

**Requirements**: `pip install model-clinic[hf]`

The callback:
- Creates a `ClinicMonitor` at `on_train_begin`
- Calls `monitor.step()` at every `on_step_end`
- Prints a summary at `on_train_end`
- Automatically forwards metrics to wandb or mlflow if active

In [ ]:
# Check if transformers is available
try:
    import transformers
    HAS_TRANSFORMERS = True
    print(f"transformers version: {transformers.__version__}")
except ImportError:
    HAS_TRANSFORMERS = False
    print("transformers not installed. Install with: pip install model-clinic[hf]")
    print("Showing callback usage pattern below without running it.")

In [ ]:
# Usage pattern — run this if transformers is installed
CALLBACK_EXAMPLE = '''
from transformers import Trainer, TrainingArguments
from model_clinic import ClinicTrainerCallback

# Create callback — check every 100 steps, alert on gradient issues
clinic_cb = ClinicTrainerCallback(
    log_every=100,
    alert_on=["gradient_explosion", "neuron_death", "layer_collapse"],
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="checkpoints",
        num_train_epochs=3,
        per_device_train_batch_size=8,
    ),
    train_dataset=train_dataset,
    callbacks=[clinic_cb],   # <-- just add it here
)

trainer.train()
# At training end, ClinicMonitor summary prints automatically
'''

print(CALLBACK_EXAMPLE)

In [ ]:
# If transformers IS available, we can demonstrate the callback on a minimal synthetic Trainer run
if HAS_TRANSFORMERS:
    from model_clinic import ClinicTrainerCallback
    from transformers import Trainer, TrainingArguments
    from torch.utils.data import Dataset

    class TinyDataset(Dataset):
        def __init__(self, n=64, dim=64):
            self.x = torch.randn(n, dim)
            self.y = torch.randn(n, dim)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            # Trainer expects dict with 'labels' for loss computation
            return {"input_ids": self.x[idx], "labels": self.y[idx]}

    print("Note: ClinicTrainerCallback works with any HuggingFace-compatible model.")
    print("See the code pattern above for the recommended integration.")
else:
    print("Skipping live HF Trainer demo (transformers not installed).")

## 8. Monitor-Only Conditions (vs Post-Hoc)

Some issues are best caught during training because they happen transiently:

| Problem | Detectable during training? | Detectable post-hoc? |
|---------|----------------------------|---------------------|
| Gradient explosion | Yes — `gradient_explosion` alert fires | Only if it caused NaN weights |
| Vanishing gradients | Yes — `gradient_vanishing` alert | No — grads are gone |
| Layer collapse (transient) | Yes — forward hook catches it | Maybe — dead neurons remain |
| Loss spike / bad batch | Yes — `loss_spike` alert | No — averaged away |
| Norm drift | Catches it early | Yes — static `norm_drift` detector |
| Dead neurons | Catches them forming | Yes — static `dead_neurons` detector |

**Rule of thumb**: Use `ClinicMonitor` during training for gradient/loss issues; use `diagnose()` post-hoc for weight-level issues.

## 9. Filtering Alerts

In large training runs you may only want alerts for critical conditions to avoid log noise. Use `alert_on` to filter.

In [ ]:
# Only alert on critical issues
critical_only = ClinicMonitor(
    TinyMLP(),
    log_every=100,
    alert_on=["gradient_explosion", "neuron_death", "layer_collapse"],
)

# Alert on everything except loss spikes (too noisy early in training)
no_loss_spike = ClinicMonitor(
    TinyMLP(),
    log_every=100,
    alert_on=[
        "gradient_explosion", "gradient_vanishing",
        "neuron_death", "weight_divergence", "layer_collapse",
    ],
)

print("ClinicMonitor instances created with filtered alert conditions.")
print("Use alert_on=None (default) to receive all alerts.")

## Summary

```python
# Minimal training loop integration
from model_clinic import ClinicMonitor

monitor = ClinicMonitor(model, log_every=100)

for step, batch in enumerate(dataloader):
    loss = train_step(batch)
    alerts = monitor.step(step, loss=loss.item())

    if any(a.severity == "ERROR" for a in alerts):
        print("Critical training error — stopping early")
        break

# After training
summary = monitor.summary()
print(f"Total alerts: {summary.total_alerts}")
```

## Next Steps

- `docs/treatment_recipes.md` — What to do after training if the monitor fired
- `docs/troubleshooting.md` — When alerts seem wrong (false positives, etc.)
- **Notebook 01** — Quickstart: post-training static analysis on checkpoints
- **Notebook 02** — Treatment Guide: fix the problems the monitor found